# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re

# путь к JSON-файлу внутри датасета
file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "Cornell-University/arxiv",
    file_path,
    pandas_kwargs={
        "lines": True,
        "nrows": 50000
    },
)


/tmp/ipykernel_8074/3707094043.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'arxiv' dataset.


In [ ]:
# заглянем
df.head(2)
len(df)

50000

### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

In [ ]:
#В пдф много артефактов, чистим датасет
# приведение id к строке фиксированной длины
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

def clean_text(text):
    """Очистка текста от лишних символов"""
    if pd.isna(text):
        return ""
    text = str(text)
    # Удаляем лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text)
    # Удаляем специальные символы, но сохраняем знаки препинания
    text = re.sub(r'[^\w\s\.\,\!\?\-\:\;\(\)\[\]\{\}]', ' ', text)
    return text.strip()

# фильтруем статьи по категории Machine Learning
# Категории в arxiv обычно имеют формат cs.LG, stat.ML и т.д.
# Будем искать по разным вариантам
filtered_df = df[
    df["categories"].str.contains("cs.LG|stat.ML|Machine Learning",
                                  na=False,
                                  case=False,
                                  regex=True)
]

# дополнительно можно ограничить по длине абстракта (убираем мусор)
filtered_df = filtered_df[filtered_df["abstract"].str.len() > 200]

# берем ровно 100 статей
filtered_df = filtered_df.head(100)

# получаем список id (ровно 100)
ids = filtered_df["id"].tolist()

# проверяем количество
print(f"Количество статей с категорией Machine Learning: {len(ids)}")

# выводим первые 5 ID для проверки
print("\nПервые 5 ID:")
for i, paper_id in enumerate(ids[:5], 1):
    print(f"{i}. {paper_id}")

# пример ссылок для первой статьи
if len(ids) > 0:
    sample_id = ids[0]
    print(f"\nПример ссылок для статьи {sample_id}:")
    abs_url = f"https://arxiv.org/abs/{sample_id}"
    pdf_url = f"https://arxiv.org/pdf/{sample_id}"
    print("Abstract page:", abs_url)
    print("PDF link:", pdf_url)

print(f"\nИтоговое количество ID: {len(ids)}")

Количество статей с категорией Machine Learning: 100

Первые 5 ID:
1. 0704.0671
2. 0704.0954
3. 00704.102
4. 0704.1028
5. 0704.1139

Пример ссылок для статьи 0704.0671:
Abstract page: https://arxiv.org/abs/0704.0671
PDF link: https://arxiv.org/pdf/0704.0671

Итоговое количество ID: 100


### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
!pip install llama-index-embeddings-langchain

In [ ]:
#загружаем абстракты, т.к. без них нейросеть не будет считывать данные, и контекста не будет
from langchain_core.documents import Document

documents = []

for idx, row in filtered_df.iterrows():
    arxiv_id = row['id']
    abstract = row['abstract']
    title = row['title']

    # Создаем документ с правильной структурой
    doc = Document(
        page_content=f"Title: {title}\n\nAbstract: {abstract}",
        metadata={
            "arxiv_id": arxiv_id,
            "title": title,
            "source": f"https://arxiv.org/abs/{arxiv_id}"
        }
    )
    documents.append(doc)

len(documents)

100

## Разбить на чанки

In [ ]:
!pip install langchain langchain_text_splitters -q
!pip install llama-index-core llama-index-embeddings-huggingface llama-index-llms-openrouter -q
!pip install langchain-openai langgraph -q
!pip install datasets pandas -q
!pip install llama-index-llms-langchain -q
!pip install langchain-core langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 7.3 MB/s eta 0:00:00


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Создаём сплиттер: размер чанка 25 символов, перекрытие 10
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=100
)

# Разбиваем текст
chunks = text_splitter.split_documents(documents)

## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install -q langchain_openai

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os
!pip install faiss-cpu sentence-transformers langchain-huggingface -q


from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.2 MB/s eta 0:00:00


In [ ]:
for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1}: {chunk}")

Чанк 1: page_content='Title: Learning from compressed observations' metadata={'arxiv_id': '0704.0671', 'title': 'Learning from compressed observations', 'source': 'https://arxiv.org/abs/0704.0671'}
Чанк 2: page_content='Abstract:   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable' metadata={'arxiv_id': '0704.0671', 'title': 'Learning from compressed observations', 'source': 'https://arxiv.org/abs/0704.0671'}
Чанк 3: page_content='i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the' metadata={'arxiv_id': '0704.0671', 'title': 'Learning from compressed observations', 'source': 'https://arxiv.org/abs/0704.0671'}
Чанк 4: page_content='asymptoti

In [ ]:
from llama_index.core import VectorStoreIndex, Document #Чтобы оформить данные в виде векторов
from llama_index.core.vector_stores import SimpleVectorStore #Чтобы оформить данные в виде векторов
from llama_index.embeddings.huggingface import HuggingFaceEmbedding #Чтобы сделать эмбеддинг модель

In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="ai-forever/sbert_large_nlu_ru"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
from llama_index.core import Document

documents = []
for a in filtered_df.iterrows():
    doc = Document(
        text=row["abstract"],  # or whatever text field you're using
        metadata={"title": row.get("title", ""), "id": row["id"]},
        id_=row["id"]  # Explicitly set the id_
    )
    documents.append(doc)

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
API3 = userdata.get('API3')

# Создаём объект LLM через LangChain
llm = ChatOpenAI(
    api_key= API3,
    base_url="https://openrouter.ai/api/v1",
    model="arcee-ai/trinity-mini:free", # бесплатная модель
)


In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model
)
retriever = index.as_retriever(similarity_top_k=4)



## Реализовать цепочку

In [ ]:
!pip install langchain_core langchain_classic -q

In [ ]:
import json
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate

In [ ]:

from google.colab import userdata
API3 = userdata.get('API3')

# Название модели
GEN_MODEL_ID = "arcee-ai/trinity-mini:free"  #наша модель

# Ставим 4 чанка для баланса. Больше 8 = галлюцинации, Меньше 4 = модель недоучена
TOP_K = 4

# Вопрос, на который хочешь получить ответ
QUESTION = "Зачем нужен Machine Learning?"

# Системный промпт которому будем скармливать чанки рага
PROMPT = """Ты — эксперт в области искусственного интеллекта и машинного обучения.
Используй только информацию из предоставленного контекста.
Если в контексте нет достаточной информации для ответа — честно скажи об этом.

Контекст:
{context}

Вопрос: {input}

Ответ:"""

print(f"Модель: {GEN_MODEL_ID}")
print(f"TOP_K: {TOP_K}")
PROMPT = PromptTemplate.from_template(PROMPT)

Модель: arcee-ai/trinity-mini:free
TOP_K: 4


In [ ]:
def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [ ]:
llm = ChatOpenAI(
    api_key=API3,
    base_url="https://openrouter.ai/api/v1",
    model="arcee-ai/trinity-mini:free", # бесплатная модель
)


In [ ]:
# Настраиваем LLM в глобальных настройках LlamaIndex
from llama_index.core import Settings

Settings.llm = llm  # используем ту же модель из OpenRouter
Settings.chunk_size = 512  # можно задать размер чанка (опционально)

# Создаём query_engine на основе существующего индекса
# Engine сам выполнит поиск чанков, сформирует промпт и вызовет LLM
query_engine = index.as_query_engine()

# Задаём тот же вопрос
query = "Зачем нужен Machine Learning?"

In [ ]:
# Получаем ответ
response = query_engine.query(query)
response

Response(response='\n\nMachine learning is needed toreconstruct dependency structures from samples, as demonstrated by the algorithm in the context. This approach allows for the reconstruction of the underlying graph defining a Markov random field using a specific number of samples based on parameters related to local interactions. The method ensures high-probability reconstruction under mild conditions and provides explicit running time bounds, making it effective for modeling complex distributions.', source_nodes=[NodeWithScore(node=TextNode(id_='fbdd7bac-4183-4d3c-bac5-207961272afa', embedding=None, metadata={'title': 'Reconstruction of Markov Random Fields from Samples: Some Easy\n  Observations and Algorithms', 'id': '0712.1402'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='0712.1402', node_type='4', metadata={'title': 'Reconstruction of Markov Random Fields from Samples: Some Easy\n  Obser

In [ ]:
print(f"Type of chunks: {type(chunks)}")
if len(chunks) > 0:
    print(f"Type of first chunk: {type(chunks[0])}")
    print(f"First chunk content: {chunks[0]}")
len(chunks)

## Протестировать на нескольких примерах, оченить качество



---



In [ ]:

from IPython.display import display, Markdown

# Список тестовых вопросов (минимум 3–5 по критериям)
test_questions = [
    # 1. Фактический вопрос (прямой факт из статей)
    "Что такое Markov random fields и для чего они используются?",

    # 2. Обобщающий вопрос (требует синтеза из нескольких статей)
    "Какие основные проблемы возникают при реконструкции графов в высокомерных распределениях?",

    # 3. Уточняющий / технический вопрос
    "Какой алгоритм предлагается для реконструкции графа Markov random field и сколько сэмплов нужно в худшем случае?",
]

results = []

for i, question in enumerate(test_questions, 1):
    print(f" Вопрос {i}/{len(test_questions)}: {question}")

    # вызываем rag
    response = query_engine.query(question)

    #  результат
    print(Markdown(f"**Ответ:**\n\n{response.response}"))

    # Показываем источники (чтобы оценить retrieval)
    print(f"\n Использованные источники ({len(response.source_nodes)} чанков):")
    for j, node in enumerate(response.source_nodes[:3], 1):  # показываем до 3
        title = node.node.metadata.get('title', 'No title')
        score = node.score
        чанк = node.node.text[:300] + "..." if len(node.node.text) > 300 else node.node.text
        print(f"  {j}. [{score:.3f}] {title[:80]}...")
        print(f"     Фрагмент: {чанк}\n")


    # Сохраняем для анализа
    results.append({
        "question": question,
        "answer": response.response,
        "sources_count": len(response.source_nodes),
        "source_nodes": response.source_nodes
    })

print("ВСЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕЕ")

 Вопрос 1/3: Что такое Markov random fields и для чего они используются?


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774569600000'}, 'provider_name': None}}, 'user_id': 'user_39pzxHp7yB1m8VAJN1fQk1BRfJK'}

In [ ]:
По поводу модели, первое что бросается в глаза это чанки. Они крошечные, их стоило бы увеличить, чтобы контекстное окно было более релевантным. Это невозможно в проекте, потому что у меня слабый компьютер

По поводу модели, первое что бросается в глаза это чанки. Они крошечные, их стоило бы увеличить, чтобы контекстное окно было более релевантным. Это невозможно в проекте, потому что у меня слабый компьютер.

Во вторых, интересно было бы увеличить сам датасет. Был кейс когда при параметрах 5000 nrows не находилось ни одной статьи по темам "AI" и "RAG"

В третьих, мы могли бы добавить эвристику, как было изначально, но у меня буквально кончились токены бесплатные на опен роутере:

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774569600000'}, 'provider_name': None}}, 'user_id': 'user_39pzxHp7yB1m8VAJN1fQk1BRfJK'}

Очень круто, я сражался с питоном 15 часов в сумме, и горжусь собой, все работает, вот так вот.

# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
